# Phase 5 — Evaluation

Compute metrics, generate visualizations, and produce the evaluation report.

**Kernel:** `gtx-1080-IP`

In [ ]:
import sys, os
from pathlib import Path

PROJECT_ROOT = Path(os.getcwd()).parent if 'notebooks' in os.getcwd() else Path(os.getcwd())
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
print(f'Project root: {PROJECT_ROOT}')

## Configuration

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

from src.utils.config import load_config, PROJECT_ROOT as PROJ_ROOT
from src.utils.seed import set_seed
from src.utils.logging import print_log

CONFIG = 'dev'
CHECKPOINT = 'checkpoints/best_model.pth'
SPLIT = 'val'  # 'val' or 'test'
SKIP_INFERENCE = False  # True if predictions already exist

cfg = load_config(CONFIG)
set_seed(cfg['seed'])
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Config: {CONFIG}, Device: {device}')

## Run Inference (if needed)

In [ ]:
from src.data.dataset import discover_brats_samples
from src.data.splits import create_splits
from src.inference.predict import run_inference

samples = discover_brats_samples(cfg['paths']['data_root'])
_, val_samples, test_samples = create_splits(
    samples,
    ratios=cfg['data']['split_ratios'],
    seed=cfg['seed'],
    splits_dir=cfg['paths']['splits_dir'],
)

target_samples = val_samples if SPLIT == 'val' else test_samples

if not SKIP_INFERENCE:
    checkpoint_path = str(PROJ_ROOT / CHECKPOINT)
    results = run_inference(
        cfg=cfg,
        checkpoint_path=checkpoint_path,
        samples=target_samples,
        device=device,
    )
else:
    print('Skipping inference — loading existing predictions')
    import nibabel as nib
    from monai.data import Dataset
    from src.data.transforms import get_val_transforms
    from src.data.dataset import get_monai_file_list
    
    file_list = get_monai_file_list(target_samples)
    ds = Dataset(data=file_list, transform=get_val_transforms(cfg))
    results = []
    
    pred_dir = Path(cfg['paths']['predictions_dir'])
    for idx, data in enumerate(ds):
        pid = target_samples[idx]['patient_id']
        pred_path = pred_dir / f'{pid}_pred.nii.gz'
        if not pred_path.exists():
            continue
        pred_seg = nib.load(str(pred_path)).get_fdata().astype(np.uint8)
        pred_mc = np.zeros((3, *pred_seg.shape), dtype=np.uint8)
        pred_mc[0] = ((pred_seg == 1) | (pred_seg == 2) | (pred_seg == 4)).astype(np.uint8)
        pred_mc[1] = ((pred_seg == 1) | (pred_seg == 4)).astype(np.uint8)
        pred_mc[2] = (pred_seg == 4).astype(np.uint8)
        results.append({'patient_id': pid, 'prediction': pred_mc, 'label': data['label'].numpy()})

print(f'{len(results)} cases ready for evaluation')

## Compute Metrics

In [ ]:
from src.evaluation.metrics import compute_case_metrics, aggregate_metrics, CLASS_NAMES

all_case_metrics = []
patient_ids = []

for r in results:
    metrics = compute_case_metrics(r['prediction'], r['label'])
    all_case_metrics.append(metrics)
    patient_ids.append(r['patient_id'])
    
    print(f'{r["patient_id"]}: '
          f'Dice WT={metrics["dice"][0]:.4f} '
          f'TC={metrics["dice"][1]:.4f} '
          f'ET={metrics["dice"][2]:.4f}')

agg = aggregate_metrics(all_case_metrics)

print(f'\n--- Aggregated ---')
for c, name in enumerate(CLASS_NAMES):
    print(f'  {name}: '
          f'Dice={agg[f"dice/{name}/mean"]:.4f}\u00b1{agg[f"dice/{name}/std"]:.4f}  '
          f'HD95={agg[f"hausdorff95/{name}/mean"]:.2f}\u00b1{agg[f"hausdorff95/{name}/std"]:.2f}')
print(f'  Mean Dice (all): {agg["dice/mean_all"]:.4f}')

## Dice Bar Chart

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

x = np.arange(len(patient_ids))
width = 0.25

for c, (name, color) in enumerate(zip(['WT', 'TC', 'ET'], ['#4CAF50', '#2196F3', '#F44336'])):
    dice_vals = [m['dice'][c] for m in all_case_metrics]
    ax.bar(x + c * width, dice_vals, width, label=name, color=color, alpha=0.8)

ax.set_xlabel('Patient')
ax.set_ylabel('Dice Coefficient')
ax.set_title('Per-Case Dice Coefficients')
ax.set_xticks(x + width)
ax.set_xticklabels(patient_ids, rotation=45, ha='right')
ax.legend()
ax.set_ylim(0, 1)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## Generate Visualizations

In [ ]:
from src.evaluation.visualize import create_overlay_figure, create_multi_view_figure
from src.data.dataset import get_monai_file_list
from src.data.transforms import get_val_transforms
from monai.data import Dataset

vis_dir = Path(cfg['paths']['visualizations_dir'])
vis_dir.mkdir(parents=True, exist_ok=True)

num_vis = cfg['evaluation'].get('num_visualization_cases', 5)

for i, r in enumerate(results[:num_vis]):
    # Load image data for this case
    file_list = get_monai_file_list([target_samples[i]])
    ds = Dataset(data=file_list, transform=get_val_transforms(cfg))
    image = ds[0]['image'].numpy()
    
    fig = create_overlay_figure(
        image=image, gt=r['label'], pred=r['prediction'],
        patient_id=r['patient_id'],
        save_path=str(vis_dir / f'{r["patient_id"]}_overlay.png'),
    )
    plt.show()
    plt.close(fig)

print(f'Visualizations saved to {vis_dir}')

## Generate Report

In [ ]:
from src.evaluation.report import generate_report

report_path = Path(cfg['paths']['output_dir']) / 'evaluation_report.md'

generate_report(
    aggregated_metrics=agg,
    case_metrics=all_case_metrics,
    patient_ids=patient_ids,
    visualization_dir=str(vis_dir),
    output_path=str(report_path),
)

print(f'\n✅ Evaluation complete.')
print(f'  Report: {report_path}')